Pset 6

In [1]:
import pandas as pd
import numpy as np

In [2]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [3]:
import matplotlib.pyplot as plt

In [4]:
import statsmodels.formula.api as smf

In [5]:
import seaborn as sns

In [6]:
from linearmodels import IV2SLS


In [7]:
import scipy.io
mat = scipy.io.loadmat('../data/jacob_lefgren.mat')

In [8]:
data = pd.DataFrame(np.array([mat["fail"], mat["summer"], mat["rdge0"], mat["mtge0"],  
                              mat["rdge1"], mat["mtge1"], mat["rmath1"], mat["rread1"], 
                              mat["rmath0"], mat["rread0"], mat["male"], mat["black"], 
                              mat["cohort"]
                             ]).reshape([13, 70831])).transpose()

In [9]:
data.columns = ['fail', 'summer', 'rdge0', 'mtge0', 'rdge1', 'mtge1', 'rmath1', 'rread1',
                'rmath0', 'rread0', 'male', 'black', 'cohort'
               ]

a)

In [10]:
threshold = 2.75

In [11]:
variable = 'running'

In [12]:
outcome = 'rmath1'

In [13]:
treatment = 'summer'

In [14]:
data[variable] = data[['mtge0', 'rdge0']].min(axis = 1)

In [15]:
data['elegible'] = (data[variable] < threshold) * 1

In [16]:
data['elegible'].mean()

0.4221315525687905

In [17]:
data.groupby('elegible')['summer'].mean()

elegible
0    0.004056
1    0.936923
Name: summer, dtype: float64

Less than 1% of non-elegible students participates on summer school. More 90% of students who are elegible participates in summer schools. We need to estimate a fuzzy RDD considering that not all elegible participants received the treatment, but the increase in the probability of enrolling into summer school is higher given the threshold.



b)

Implementing the optimal bandwith estimator

In [18]:

def get_optimal_bandwith(data, variable, outcome, threshold):

    # Step 1
    
    N = data.shape[0]
    
    s2_X = data[variable].var()*N/(N - 1)
    
    h1 = 1.84 * s2_X ** 0.5 / (N **0.2)
    
    #Step 2
    
    sample_left = data[(data[variable] > threshold - h1) & (data[variable] < threshold)]
    sample_right = data[(data[variable] < threshold + h1) & (data[variable] > threshold)]
    
    bar_Y_left_h1 = sample_left[outcome].mean()
    bar_Y_right_h1 = sample_right[outcome].mean()
    
    N_h1_left = sample_left.shape[0]
    N_h1_right = sample_right.shape[0]
    
    f_c = (N_h1_left + N_h1_right) / (2 * N * h1)
    
    sigma2_left = sample_left[variable].var() * N_h1_left/(N_h1_left - 1)
    sigma2_right = sample_right[variable].var() * N_h1_right/(N_h1_right - 1)
    
    # Step 3
    
    
    
    model = smf.ols(f"{outcome} ~ 1 + I({variable} - {threshold}) + I(({variable} > {threshold})) + I(({variable} - {threshold}) ** 2) + I(({variable} - {threshold})** 3)", data = data).fit()   
    
    m3_c = model.params.iloc[-1]
    
    N_right = (data[variable] > threshold).sum()
    N_left = (data[variable] < threshold).sum()
    
    h2_right = 3.56 * (sigma2_right/(f_c * (m3_c ** 2))) ** (1/7) * (N_right ** (-1/7))
    
    h2_left = 3.56 * (sigma2_left/(f_c * (m3_c ** 2))) ** (1/7) * (N_left ** (-1/7))
    
    # Curvature estimation
    
    sample_left_h2 = data[(data[variable] < threshold) & (data[variable] > threshold - h2_left)]
    sample_right_h2 = data[(data[variable] > threshold) & (data[variable] < threshold + h2_right)]
    
    model_left = smf.ols(f"{outcome} ~ 1 + I({variable} - {threshold}) + I(({variable} - {threshold}) ** 2)", data = sample_left_h2).fit()
    model_right = smf.ols(f"{outcome} ~ 1 + I({variable} - {threshold}) + I(({variable} - {threshold}) ** 2)", data = sample_right_h2).fit()
    
    N2_right =sample_right_h2.shape[0]
    N2_left = sample_left_h2.shape[0]
    
    m2_right = 2 * model_right.params.iloc[-1]
    m2_left = 2 * model_left.params.iloc[-1]
    
    # Regularization
    
    r_right = 2160 * sigma2_right / (N2_right * (h2_right ** 4))
    
    r_left = 2160 * sigma2_left / (N2_left * (h2_left ** 4))
    
    # Optimal bandwith
    
    h_opt_edge = 3.4375 * ((sigma2_left + sigma2_right) / (f_c * ((m2_right - m2_left) ** 2) + r_left + r_right))**(1/5) * (N ** (-1/5))
    h_opt_unif = 5.40 * ((sigma2_left + sigma2_right) / (f_c * ((m2_right - m2_left) ** 2) + r_left + r_right))**(1/5) * (N ** (-1/5))

    return h_opt_edge, h_opt_unif

In [19]:
h_opt_edge, h_opt_unif = get_optimal_bandwith(data, variable, outcome, threshold)

In [20]:
h_opt = h_opt_unif

In [21]:
h_opt_edge

0.6224706928887828

In [22]:
h_opt_unif

0.9778448702834698

Estimating the fuzzy RDD considering an h_opt, for linear and quadratic specification

In [23]:
def estimate_frdd(data, variable, treatment, outcome, h_opt, threshold):

    m2s = IV2SLS.from_formula(f"{outcome} ~ 1 + [{treatment}~I({variable} > {threshold})] + I({variable} - {threshold}) + I({variable} - {threshold}):I({variable} > {threshold})", 
                          data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit(cov_type = 'robust')

    m2s_q = IV2SLS.from_formula(f"{outcome} ~ 1 + [{treatment}~I({variable} > {threshold})] + I({variable} - {threshold}) + I({variable} - {threshold}):I({variable} > {threshold}) + I(({variable} - {threshold}) ** 2) + I(({variable} - {threshold})**2):I({variable} > {threshold})", 
                          data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit(cov_type = 'robust')

    return m2s, m2s_q


Estimating fuzzy RDD (step by step)

In [24]:
fs_k = smf.ols(f"{treatment} ~ I({variable} > {threshold}) + I({variable} - {threshold}) + I({variable} - {threshold}):I({variable} > {threshold})", 
                    data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit()

In [25]:
rf_k = smf.ols(f"{outcome} ~ I({variable} > {threshold}) + I({variable} - {threshold}) + I({variable} - {threshold}):I({variable} > {threshold})", 
                    data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit()

In [26]:
rf_k.params['I(running > 2.75)[T.True]']

-0.1466697915712528

In [27]:
fs_k.params['I(running > 2.75)[T.True]']

-0.9098910568804895

In [28]:
rdd_est = rf_k.params['I(running > 2.75)[T.True]'] / fs_k.params['I(running > 2.75)[T.True]']

In [29]:
rdd_est

0.16119489301730466

Checking quadratic specification, to remove potential biases

In [30]:
fs_q = smf.ols(f"{treatment} ~ I({variable} > {threshold}) + I({variable} - {threshold}) + I(({variable} - {threshold}) ** 2) + I({variable} - {threshold}):I({variable} > {threshold})+ I(({variable} - {threshold})**2):I({variable} > {threshold})", 
                    data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit()

In [31]:
rf_q = smf.ols(f"{outcome} ~ I({variable} > {threshold}) + I({variable} - {threshold}) + I(({variable} - {threshold}) ** 2) + I({variable} - {threshold}):I({variable} > {threshold})+ I(({variable} - {threshold})**2):I({variable} > {threshold})", 
                    data = data[(data[variable] - threshold).apply(abs) < h_opt]).fit()

In [32]:
rdd_est_q = rf_q.params['I(running > 2.75)[T.True]'] / fs_q.params['I(running > 2.75)[T.True]']

In [33]:
fs_q.params['I(running > 2.75)[T.True]']

-0.8942214965870494

In [34]:
rf_q.params['I(running > 2.75)[T.True]']

-0.15802655577353614

In [35]:
rdd_est

0.16119489301730466

Implementation

In [36]:
def get_results(model, treatment, variable, threshold):

    return (model.params[treatment], 
            model.std_errors[treatment], 
            model.first_stage.individual[treatment].params[f'I({variable} > {threshold})'])

In [37]:
def get_graph(data, variable, threshold, h_opt, variable_check, label):

    
    # Graph for linear - Edge
    sns.regplot(data = data[(data[variable] >= threshold - h_opt) & (data[variable] <= threshold)], x = variable, y = variable_check, x_bins = 5, order = 1, color = 'red')
    sns.regplot(data = data[(data[variable] >= threshold - h_opt) & (data[variable] <= threshold)], x = variable, y = variable_check, x_bins = 5, order = 2, color = 'gray')
    
    sns.regplot(data = data[(data[variable] <= threshold + h_opt) & (data[variable] > threshold)], x = variable, y = variable_check, x_bins = 5, order = 1, color = 'red')
    sns.regplot(data = data[(data[variable] <= threshold + h_opt) & (data[variable] > threshold)], x = variable, y = variable_check, x_bins = 5, order = 2, color = 'gray')

    plt.axvline(threshold)
    
    plt.savefig(f"graph_rdd_{label}.png")

    plt.close()
    
    




In [38]:
def get_estimates(data, variable, treatment, outcome, threshold):
    h_opt_edge, h_opt_unif = get_optimal_bandwith(data, variable, outcome, threshold)    
    print("Edge kernel")
    print(h_opt_edge)
    print("Uniform kernel")
    print(h_opt_unif)
    
    l_edge, q_edge = estimate_frdd(data, variable, treatment, outcome, h_opt_edge, threshold)
    
    l_unif, q_unif = estimate_frdd(data, variable, treatment, outcome, h_opt_unif, threshold)
    
    estimates = pd.DataFrame(map(lambda model: get_results(model, treatment, variable, threshold), 
                                 [l_unif, q_unif, l_edge, q_edge]), 
                             columns = ['estimate', 'se', 'first_stage'], 
                             index = ['Linear - Uniform Kernel', 'Quadratic - Uniform Kernel',
                                      'Linear - Edge Kernel', 'Quadratic - Edge Kernel']
                            )

    estimates['reduce_form'] = estimates['estimate']/estimates['first_stage']
    
    get_graph(data, variable, threshold, h_opt_edge, treatment, f"{variable}_{treatment}_edge")
    get_graph(data, variable, threshold, h_opt_edge, outcome, f"{variable}_{outcome}_edge")

    get_graph(data, variable, threshold, h_opt_unif, treatment, f"{variable}_{treatment}_uni")
    get_graph(data, variable, threshold, h_opt_unif, outcome, f"{variable}_{outcome}_uni")
    
    
                                                                                                                                                            
    return estimates

The interpretation of our results should be considered as a LATE, considering that the threshold is a string incentive to enroll into summer school. Therefore, the estimated effect should be interpreted as the effect on compliers, students who will attend summer school if they got less than 2.75 in any of the both scores, reading or math scores.

In [39]:
get_estimates(data, variable, treatment, "male", threshold)

Edge kernel
0.4822965169331964
Uniform kernel
0.7576439829641486


,estimate,se,first_stage,reduce_form
Linear - Uniform Kernel,-0.007019,0.010525,-0.905595,0.007750
Quadratic - Uniform Kernel,-0.004538,0.016543,-0.892522,0.005084
Linear - Edge Kernel,-0.002185,0.013814,-0.900449,0.002426
Quadratic - Edge Kernel,0.004328,0.022436,-0.873976,-0.004952


In [40]:
get_estimates(data, variable, treatment, "black", threshold)

Edge kernel
0.4486399421545917
Uniform kernel
0.7047725636755767


,estimate,se,first_stage,reduce_form
Linear - Uniform Kernel,-0.016666,0.009506,-0.904507,0.018426
Quadratic - Uniform Kernel,-0.027412,0.015104,-0.889512,0.030817
Linear - Edge Kernel,-0.031750,0.012868,-0.893220,0.035546
Quadratic - Edge Kernel,-0.027956,0.021832,-0.878667,0.031816


The interpretation of our results should be considered as a LATE, considering that the threshold is a string incentive to enroll into summer school. Therefore, the estimated effect should be interpreted as the effect on compliers, students who will attend summer school if they got less than 2.75 in any of the both scores, reading or math scores.

In [41]:
get_estimates(data, variable, treatment, outcome, threshold)

Edge kernel
0.6224706928887828
Uniform kernel
0.9778448702834698


,estimate,se,first_stage,reduce_form
Linear - Uniform Kernel,0.161195,0.012959,-0.909891,-0.177158
Quadratic - Uniform Kernel,0.176720,0.019817,-0.894221,-0.197624
Linear - Edge Kernel,0.148413,0.016470,-0.901659,-0.164600
Quadratic - Edge Kernel,0.276184,0.026063,-0.887835,-0.311075


c)

Since the estimates we get should be interpreted as local, and in this case our instrument is the math score, we should considere this effecct as the effect on compliers under this specification, students who will attend summer school if they got less than 2.75 in math scores compared to students who wouldn't atend summer school is they got more than 2.75. Since there is a chance of students of still enroll in summer school because of the reading test, this result is not comparable with the one we got in the previous specification.

In [42]:
get_estimates(data, "mtge0", treatment, outcome, threshold)

Edge kernel
0.49462526901187187
Uniform kernel
0.7770113316841043


,estimate,se,first_stage,reduce_form
Linear - Uniform Kernel,0.185609,0.043368,-0.294229,-0.630831
Quadratic - Uniform Kernel,0.128578,0.069976,-0.278468,-0.461733
Linear - Edge Kernel,0.139615,0.056620,-0.282707,-0.493848
Quadratic - Edge Kernel,0.131916,0.089704,-0.285597,-0.461895


d)

Since the estimates we get should be interpreted as local, and since in this case our instrument is the reading score, we should considere this effect as the effect on compliers under this specification, students who will attend summer school if they got less than 2.75 in the reading scores compared to students who wouldn't atend summer school is they got more than 2.75 in this same score. Since there is a chance these students are still enrolling in summer school because of the math test, this result is not comparable to the one we got in the previous specification.

In [43]:
get_estimates(data, "rdge0", treatment, outcome, threshold)

Edge kernel
0.600001675433395
Uniform kernel
0.942548086499006


,estimate,se,first_stage,reduce_form
Linear - Uniform Kernel,0.120670,0.018835,-0.728940,-0.165541
Quadratic - Uniform Kernel,0.147400,0.029578,-0.707738,-0.208269
Linear - Edge Kernel,0.129178,0.023489,-0.714479,-0.180800
Quadratic - Edge Kernel,0.199581,0.036976,-0.709273,-0.281389
